# Classification & Segmentation Evaluation

Compares model predictions against CODA-class ground-truth GeoJSONs.

| Cell | Purpose |
|------|---------|
| 1 | Imports |
| **2** | **Parameters ← edit here** |
| 3 | Helper functions |
| 4 | Load GT GeoJSON |
| 5 | Load predictions or run inference |
| 6 | Match GT ↔ predictions |
| 7 | Per-class metrics |
| 8 | Confusion matrix |
| 9 | Per-class F1 bar chart |
| 10 | Save results |

**Metrics:**
- **Detection** — overall P / R / F1 (nucleus found, any class)
- **Classification** — per-class P / R / F1 on centroid-matched pairs
- **Confusion matrix** — GT class (rows) × predicted class (cols)


---
## 1 · Imports


In [ ]:
import json
import concurrent.futures
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from tqdm import tqdm
import torch
import openslide

import sys
sys.path.insert(0, str(Path.cwd()))

from inference_utils import (
    load_model_and_classes,
    build_tile_coords,
    forward_batch_with_perm,
    postprocess_batch_v4,
    get_tile_from_ram,
)
from geometry import (
    dedupe_nucleus_features_by_centroid,
    dedupe_nucleus_features_by_polygon_overlap,
)

print("Imports OK")


---
## 2 · Parameters  ← edit here


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET = "GS40"   # "GS40" | "GS55" | "GS33"

DATASET_CONFIGS = {
    "GS40": {
        "gt_dir":  Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS40/geojson_CODAclass"),
        "wsi_dir": Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS40"),
        "wsi_glob": "*.ndpi",
    },
    "GS55": {
        "gt_dir":  Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS55/geojson_CODAclass"),
        "wsi_dir": Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS55"),
        "wsi_glob": "*.ndpi",
    },
    "GS33": {
        "gt_dir":  Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS33/geojson_CODAclass"),
        "wsi_dir": Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS33"),
        "wsi_glob": "*.ndpi",
    },
}
GT_DIR   = DATASET_CONFIGS[DATASET]["gt_dir"]
WSI_DIR  = DATASET_CONFIGS[DATASET]["wsi_dir"]
WSI_GLOB = DATASET_CONFIGS[DATASET]["wsi_glob"]

# Slide to evaluate — None = pick first GT file automatically
SLIDE_STEM = None   # e.g. "monkey_fetus_40_0451"

# Path to an existing classified GeoJSON from infer_wsi_v4.  None = run inference.
PRED_GEOJSON = None
# PRED_GEOJSON = Path("//kittyserverdw/.../monkey_fetus_40_0451_classified.geojson")

# ── Model (used only when PRED_GEOJSON is None) ──────────────────────────────
WEIGHTS_PATH = Path("//kittyserverdw/Andre_kit/data/students/Diogo/data/fetal/GS40/cellvit_training/dataset_256_40k_48_slides/convnext_stardist_multitask_runs/run_gs40_gs55_v3_finetune/convnext_stardist_mt_gs40_gs55_v3_finetune/best.pt")
CONFIG_PATH  = Path("C:/Users/Andre/cursor_projects/Convnext_stardist/shared_convnext_stardist_decoder/config_finetune_gs40_gs55.yaml")

# ── Inference settings (used only when PRED_GEOJSON is None) ─────────────────
SLIDE_LEVEL         = 0
LOAD_TO_RAM         = True
TILE_SIZE           = 256
TILE_OVERLAP        = 64
VALID_MARGIN        = 32
PROB_THRESH         = 0.45
NMS_DIST            = 8
DEDUP_MIN_DIST      = 10.0
DEDUP_POLYGON       = True
DEDUP_OVERLAP_RATIO = 0.40
DEDUP_MIN_IOU       = 0.28
DEDUP_GRID_PX       = 32.0
DEVICE_STR          = "cuda"
USE_FP16            = True
BATCH_TILES         = 64
NUM_WORKERS         = 8
BG_FILTER           = True
BG_THRESHOLD        = 240
MIN_TISSUE_FRAC     = 0.10
REFINE_PEAK_COM     = True
REFINE_PEAK_RADIUS  = 8
VOTE_WINDOW_PX      = 9

# ── Evaluation ────────────────────────────────────────────────────────────────
MATCH_DIST_PX = 8.0     # max centroid distance (20x px) to count as a match
GT_IGNORE     = {"Unassigned", "OutsideMask"}

# ── Class palette (must match checkpoint) ────────────────────────────────────
LABELS_VIZ = [
    "bone",   "brain",  "eye",        "heart",     "lungs",
    "GI",     "liver",  "spleen",     "pancreas",  "kidney",
    "mesokidney", "collagen", "ear",  "nontissue", "thymus",
    "thyroid", "bladder", "skull",    "spleen2",
]
COLORS_VIZ = [
    [214,212,161],[247,184,67],[136,232,95],[140,13,13],[38,27,166],
    [13,125,11],[179,50,108],[228,235,131],[156,96,235],[46,190,230],
    [150,255,245],[254,222,255],[235,154,108],[255,255,255],[9,64,116],
    [255,255,74],[178,178,0],[214,212,161],[54,83,89],
]

DEVICE  = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")
OUT_DIR = Path.cwd()
print(f"Device  : {DEVICE}")
print(f"Dataset : {DATASET}  |  GT dir: {GT_DIR}")


---
## 3 · Helper functions


In [ ]:
def load_geojson_features(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    return data if isinstance(data, list) else data.get("features", [])


def _poly_centroid_xy(ring):
    pts = np.asarray(ring, dtype=np.float64)
    if pts.ndim == 2 and pts.shape[0] >= 2:
        return pts[:-1].mean(axis=0)   # [x, y] — drop closing duplicate
    return None


def get_class_name(feat):
    cls  = feat.get("properties", {}).get("classification", {})
    name = cls.get("name") if isinstance(cls, dict) else None
    return name.strip().lower() if name else None


def extract_centroids_and_classes(features, ignore_classes=None):
    ignore = {n.lower() for n in (ignore_classes or [])}
    centroids, classes = [], []
    for feat in features:
        name = get_class_name(feat)
        if name is None or name in ignore:
            continue
        ring = feat.get("geometry", {}).get("coordinates", [[]])[0]
        c = _poly_centroid_xy(ring)
        if c is None:
            continue
        centroids.append(c)
        classes.append(name)
    if not centroids:
        return np.zeros((0, 2)), []
    return np.array(centroids, dtype=np.float64), classes


print("Helpers defined.")


---
## 4 · Load GT GeoJSON


In [ ]:
gt_files = sorted(GT_DIR.glob("*.geojson"))
if not gt_files:
    raise FileNotFoundError(f"No .geojson files in {GT_DIR}")

if SLIDE_STEM is None:
    gt_path    = gt_files[0]
    slide_stem = gt_path.stem
else:
    slide_stem = SLIDE_STEM
    gt_path    = GT_DIR / f"{slide_stem}.geojson"
    if not gt_path.exists():
        raise FileNotFoundError(f"GT file not found: {gt_path}")

print(f"GT file  : {gt_path.name}")
print(f"Slide    : {slide_stem}")

gt_all = load_geojson_features(gt_path)
print(f"GT total : {len(gt_all):,} features")

gt_centroids, gt_classes = extract_centroids_and_classes(gt_all, ignore_classes=GT_IGNORE)
print(f"GT valid : {len(gt_centroids):,} nuclei (ignoring {GT_IGNORE})")

gt_dist = Counter(gt_classes)
print("\nGT class distribution:")
for name, cnt in sorted(gt_dist.items(), key=lambda x: -x[1]):
    print(f"  {name:20s}  {cnt:7,}  ({100*cnt/max(len(gt_classes),1):.1f}%)")


---
## 5 · Load predictions or run inference


In [ ]:
def _infer_slide(slide_path, model, device, fp16, idx2label, cls_perm=None):
    """Run full-slide inference, return list of GeoJSON features."""
    slide = openslide.OpenSlide(str(slide_path))
    W, H  = slide.level_dimensions[SLIDE_LEVEL]
    print(f"  Slide  : {slide_path.name}  {W}x{H}")
    _fp16 = fp16 and device.type == "cuda"
    model.half() if _fp16 else model.float()

    def _is_tissue(t):
        return float((t.mean(axis=2) < BG_THRESHOLD).mean()) >= MIN_TISSUE_FRAC if BG_FILTER else True

    slide_ram = None
    if LOAD_TO_RAM:
        print(f"  Loading to RAM (~{W*H*3/1e6:.0f} MB)...")
        slide_ram = np.asarray(
            slide.read_region((0, 0), SLIDE_LEVEL, (W, H)).convert("RGB"), dtype=np.uint8
        )

    tile_coords = build_tile_coords(W, H, TILE_SIZE, TILE_OVERLAP)
    print(f"  Tiles  : {len(tile_coords):,}  (overlap={TILE_OVERLAP})")

    all_features, feat_id = [], [0]
    post_kw = dict(
        nms_dist=NMS_DIST, prob_thresh=PROB_THRESH,
        refine_local_com=REFINE_PEAK_COM, refine_radius_px=REFINE_PEAK_RADIUS,
        cls_perm=cls_perm, vote_window_px=VOTE_WINDOW_PX, include_class_probs=False,
        valid_margin=VALID_MARGIN, w_slide=W, h_slide=H, idx2label=idx2label,
    )

    def _drain(fut):
        if fut is None:
            return
        for f in fut.result():
            f["id"] = str(feat_id[0])
            feat_id[0] += 1
            all_features.append(f)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max(1, NUM_WORKERS)) as pool, \
         concurrent.futures.ThreadPoolExecutor(max_workers=1) as post_pool:
        pending = None
        pbar = tqdm(total=len(tile_coords), desc="Inference", unit="tile")
        for bs in range(0, len(tile_coords), BATCH_TILES):
            meta = tile_coords[bs: bs + BATCH_TILES]
            if slide_ram is not None:
                patches = [pool.submit(get_tile_from_ram, slide_ram, x0, y0, tw, th).result()
                           for x0, y0, tw, th in meta]
            else:
                patches = [
                    np.asarray(slide.read_region((x0, y0), SLIDE_LEVEL, (tw, th)).convert("RGB"), dtype=np.uint8)
                    for x0, y0, tw, th in meta
                ]
            keep = [i for i, p in enumerate(patches) if _is_tissue(p)]
            if not keep:
                pbar.update(len(meta))
                continue
            patches = [patches[i] for i in keep]
            meta    = [meta[i]    for i in keep]
            results = forward_batch_with_perm(model, patches, device, fp16=_fp16, cls_perm=cls_perm)
            _drain(pending)
            pending = post_pool.submit(postprocess_batch_v4, list(meta), results, **post_kw)
            pbar.update(len(meta))
            pbar.set_postfix(nuclei=feat_id[0])
        _drain(pending)
        pbar.close()

    model.float()
    slide.close()
    n_raw = len(all_features)
    all_features = dedupe_nucleus_features_by_centroid(all_features, min_dist_px=DEDUP_MIN_DIST)
    if DEDUP_POLYGON:
        try:
            all_features = dedupe_nucleus_features_by_polygon_overlap(
                all_features, min_overlap_ratio=DEDUP_OVERLAP_RATIO,
                min_iou=DEDUP_MIN_IOU, grid_cell_px=DEDUP_GRID_PX,
            )
        except ImportError:
            print("  shapely not found — polygon dedup skipped")
    for i, f in enumerate(all_features):
        f["id"] = str(i)
    print(f"  Detections: {n_raw:,} -> {len(all_features):,} after dedup")
    return all_features


# ── Load or run ───────────────────────────────────────────────────────────────
if PRED_GEOJSON is not None:
    print(f"Loading predictions: {PRED_GEOJSON.name}")
    pred_all  = load_geojson_features(PRED_GEOJSON)
    idx2label = None
else:
    wsi_files = sorted(WSI_DIR.glob(WSI_GLOB))
    wsi_path  = next((p for p in wsi_files if p.stem == slide_stem), None)
    if wsi_path is None:
        wsi_path = next((p for p in wsi_files if slide_stem in p.stem), None)
    if wsi_path is None:
        raise FileNotFoundError(f"WSI for '{slide_stem}' not found in {WSI_DIR}")
    model, idx2label = load_model_and_classes(WEIGHTS_PATH, CONFIG_PATH, DEVICE)
    print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
    pred_all = _infer_slide(wsi_path, model, DEVICE, USE_FP16, idx2label)

pred_centroids, pred_classes = extract_centroids_and_classes(pred_all)
pred_dist = Counter(pred_classes)
print(f"\nPredictions: {len(pred_centroids):,} nuclei")
for name, cnt in sorted(pred_dist.items(), key=lambda x: -x[1]):
    print(f"  {name:20s}  {cnt:7,}  ({100*cnt/max(len(pred_classes),1):.1f}%)")


---
## 6 · Match GT ↔ predictions


In [ ]:
def _match_nuclei(gt_xy, pred_xy, dist_thresh):
    """Greedy bijective centroid matching (closest-first)."""
    if len(gt_xy) == 0 or len(pred_xy) == 0:
        return [], list(range(len(pred_xy))), list(range(len(gt_xy)))
    tree = cKDTree(gt_xy)
    dists, gt_idxs = tree.query(pred_xy, k=1)
    candidates = sorted(
        [(float(dists[i]), i, int(gt_idxs[i]))
         for i in range(len(pred_xy)) if dists[i] < dist_thresh]
    )
    matched, used_gt, used_pred = [], set(), set()
    for _, pred_i, gt_j in candidates:
        if gt_j not in used_gt and pred_i not in used_pred:
            matched.append((pred_i, gt_j))
            used_gt.add(gt_j)
            used_pred.add(pred_i)
    return (
        matched,
        [i for i in range(len(pred_xy)) if i not in used_pred],
        [j for j in range(len(gt_xy))   if j not in used_gt],
    )


matched_pairs, unmatched_pred, unmatched_gt = _match_nuclei(
    gt_centroids, pred_centroids, MATCH_DIST_PX
)
n_m  = len(matched_pairs)
n_fp = len(unmatched_pred)
n_fn = len(unmatched_gt)

det_p  = n_m / (n_m + n_fp + 1e-9)
det_r  = n_m / (n_m + n_fn + 1e-9)
det_f1 = 2 * det_p * det_r / (det_p + det_r + 1e-9)

print(f"Matching  threshold = {MATCH_DIST_PX} px")
print(f"  Matched        : {n_m:,}")
print(f"  Unmatched pred : {n_fp:,}  (FP — extra detections)")
print(f"  Unmatched GT   : {n_fn:,}  (FN — missed nuclei)")
print(f"\nDetection metrics (any class):")
print(f"  Precision {det_p:.4f}  Recall {det_r:.4f}  F1 {det_f1:.4f}")


---
## 7 · Per-class metrics


In [ ]:
all_classes = sorted(set(gt_classes) | set(pred_classes))
tp = Counter(); fp = Counter(); fn = Counter()

for pred_i, gt_j in matched_pairs:
    gc = gt_classes[gt_j]
    pc = pred_classes[pred_i]
    if gc == pc:
        tp[gc] += 1
    else:
        fn[gc] += 1   # found but misclassified -> FN for GT class
        fp[pc] += 1   # wrong class predicted   -> FP for pred class

for pred_i in unmatched_pred:
    fp[pred_classes[pred_i]] += 1   # FP: detection with no GT match

for gt_j in unmatched_gt:
    fn[gt_classes[gt_j]] += 1       # FN: missed GT nucleus

rows = []
for c in all_classes:
    p = tp[c] / (tp[c] + fp[c] + 1e-9)
    r = tp[c] / (tp[c] + fn[c] + 1e-9)
    f = 2 * p * r / (p + r + 1e-9)
    rows.append({"class": c, "TP": tp[c], "FP": fp[c], "FN": fn[c],
                 "precision": p, "recall": r, "F1": f,
                 "GT_count": gt_dist.get(c, 0), "pred_count": pred_dist.get(c, 0)})

df = pd.DataFrame(rows).sort_values("GT_count", ascending=False).reset_index(drop=True)
w_gt        = df["GT_count"]
macro_f1    = float(df["F1"].mean())
weighted_f1 = float((df["F1"] * w_gt).sum() / (w_gt.sum() + 1e-9))

pd.set_option("display.float_format", "{:.3f}".format)
print("Per-class metrics (sorted by GT count):")
print(df[["class","TP","FP","FN","precision","recall","F1","GT_count"]].to_string(index=False))
print(f"\nMacro F1   : {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")


---
## 8 · Confusion matrix


In [ ]:
cls_list = sorted(set(gt_classes) | set(pred_classes))
c2i      = {c: i for i, c in enumerate(cls_list)}
n_cls    = len(cls_list)

cm = np.zeros((n_cls, n_cls), dtype=np.int64)
for pred_i, gt_j in matched_pairs:
    cm[c2i[gt_classes[gt_j]], c2i[pred_classes[pred_i]]] += 1

cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(26, 10))
for ax, data, title, fmt in [
    (axes[0], cm,      "Counts",        ".0f"),
    (axes[1], cm_norm, "Row-normalised", ".2f"),
]:
    vmax = 1.0 if fmt == ".2f" else None
    im   = ax.imshow(data, cmap="Blues", vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(n_cls))
    ax.set_xticklabels(cls_list, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n_cls))
    ax.set_yticklabels(cls_list, fontsize=8)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("GT class")
    ax.set_title(title)
    thresh = data.max() * 0.6 if data.max() > 0 else 1.0
    for i in range(n_cls):
        for j in range(n_cls):
            if data[i, j] == 0:
                continue
            ax.text(j, i, f"{data[i, j]:{fmt}}", ha="center", va="center",
                    fontsize=6, color="white" if data[i, j] > thresh else "black")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(
    f"{slide_stem}  |  matched={n_m:,}  dist={MATCH_DIST_PX}px  "
    f"det F1={det_f1:.3f}  wt cls F1={weighted_f1:.3f}",
    fontsize=11,
)
plt.tight_layout()
out_cm = OUT_DIR / f"{slide_stem}_confusion_matrix.png"
plt.savefig(out_cm, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_cm}")


---
## 9 · Per-class F1 bar chart


In [ ]:
label_to_rgb = {lbl.lower(): np.array(col) / 255.0 for lbl, col in zip(LABELS_VIZ, COLORS_VIZ)}

df_sorted  = df.sort_values("F1", ascending=True)
bar_colors = [label_to_rgb.get(r["class"], np.array([0.6, 0.6, 0.6]))
              for _, r in df_sorted.iterrows()]

fig, ax = plt.subplots(figsize=(12, max(5, len(df_sorted) * 0.50)))
bars = ax.barh(df_sorted["class"].tolist(), df_sorted["F1"].tolist(), color=bar_colors)
ax.set_xlim(0, 1.18)
ax.axvline(macro_f1,    color="red",    linestyle="--", lw=1.3, label=f"Macro F1 = {macro_f1:.3f}")
ax.axvline(weighted_f1, color="orange", linestyle="--", lw=1.3, label=f"Weighted F1 = {weighted_f1:.3f}")
for bar, (_, row) in zip(bars, df_sorted.iterrows()):
    ax.text(
        bar.get_width() + 0.012, bar.get_y() + bar.get_height() / 2,
        f"P={row['precision']:.2f}  R={row['recall']:.2f}  n={int(row['GT_count']):,}",
        va="center", fontsize=8,
    )
ax.set_xlabel("F1 score")
ax.set_title(
    f"Per-class F1 -- {slide_stem}\n"
    f"det F1={det_f1:.3f}  matched={n_m:,} / GT={len(gt_centroids):,} / pred={len(pred_centroids):,}"
)
ax.legend(fontsize=9)
plt.tight_layout()
out_f1 = OUT_DIR / f"{slide_stem}_per_class_f1.png"
plt.savefig(out_f1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_f1}")


---
## 10 · Save results


In [ ]:
results = {
    "slide_stem":     slide_stem,
    "dataset":        DATASET,
    "match_dist_px":  MATCH_DIST_PX,
    "gt_nuclei":      int(len(gt_centroids)),
    "pred_nuclei":    int(len(pred_centroids)),
    "matched":        int(n_m),
    "unmatched_pred": int(n_fp),
    "unmatched_gt":   int(n_fn),
    "detection": {
        "precision": round(det_p,  6),
        "recall":    round(det_r,  6),
        "f1":        round(det_f1, 6),
    },
    "classification": {
        "macro_f1":    round(macro_f1,    6),
        "weighted_f1": round(weighted_f1, 6),
        "per_class":   df.round(4).to_dict(orient="records"),
    },
}
out_json = OUT_DIR / f"{slide_stem}_eval_metrics.json"
out_csv  = OUT_DIR / f"{slide_stem}_per_class_metrics.csv"
out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
df.to_csv(out_csv, index=False, float_format="%.4f")
print(f"Saved metrics : {out_json}")
print(f"Saved CSV     : {out_csv}")
